In [ ]:
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OrdinalEncoder

import lightgbm as lgb

In [ ]:
exp_name = '001_stacking'

In [ ]:
DATA_PATH = Path('/Users/gouyashuto/localrepository/atmacup18/input')
OUTPUT_DIR = Path('/Users/gouyashuto/localrepository/atmacup18/output')

In [ ]:
train_df = pl.read_csv(DATA_PATH / 'train_features.csv')
test_df = pl.read_csv(DATA_PATH / 'test_features.csv')

print(train_df.shape)
print(test_df.shape)

In [ ]:
CAT_COLS = ['gearShifter', 'scene']

TARGET_COLS = [
    'x_0', 'y_0', 'z_0',
    'x_1', 'y_1', 'z_1',
    'x_2', 'y_2', 'z_2',
    'x_3', 'y_3', 'z_3',
    'x_4', 'y_4', 'z_4',
    'x_5', 'y_5', 'z_5'
]

In [ ]:
def get_agg_exprs(agg_cols) -> list[pl.Expr]:
    # ????????????
    exprs = []
    exprs += [pl.col(agg_col).shift(-1).over("scene").alias(f"{agg_col}_shift-1") for agg_col in agg_cols] # 1??????????
    exprs += [pl.col(agg_col).shift(1).over("scene").alias(f"{agg_col}_shift1") for agg_col in agg_cols] # 1??????????
    exprs += [pl.col(agg_col).diff(-1).over("scene").alias(f"{agg_col}_diff-1") for agg_col in agg_cols] # 1??????????????
    exprs += [pl.col(agg_col).diff(1).over("scene").alias(f"{agg_col}_diff1") for agg_col in agg_cols] # 1??????????????
    exprs += [pl.col(agg_col).mean().over("scene").alias(f"{agg_col}_mean") for agg_col in agg_cols] # ?????????
    exprs += [pl.col(agg_col).std().over("scene").alias(f"{agg_col}_std") for agg_col in agg_cols] # ??????????
    exprs += [pl.col(agg_col).max().over("scene").alias(f"{agg_col}_max") for agg_col in agg_cols] # ?????????
    exprs += [pl.col(agg_col).min().over("scene").alias(f"{agg_col}_min") for agg_col in agg_cols] # ?????????
    return exprs

In [ ]:
def preprocess(df: pl.DataFrame) -> pl.DataFrame:
    agg_cols = ['vEgo', 'aEgo', 'steeringAngleDeg', 'steeringTorque', 'gas'] # ?????????????????
    df = (
        df
        .with_columns(
            scene = pl.col('ID').str.split('_').list[0],
            decisecond = pl.col('ID').str.split('_').list[1].cast(pl.Int32),
        )
        .sort(['scene', 'decisecond'])
        .with_columns(get_agg_exprs(agg_cols))
    )
    return df

In [ ]:
def train(X: pd.DataFrame, y: pd.DataFrame, target: str):
    lgb_params = {
        'objective': 'regression',
        'metric': 'mae',
        'learning_rate': 0.01,
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'n_jobs': -1,
    }
    gkf = GroupKFold(n_splits=5)
    groups = X['scene']

    models = []
    oof = np.zeros(len(X))
    # for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y[target], groups=groups)):
    for fold in range(5):
        print('=' * 10, f'fold: {fold} start' + '=' * 10)
        train_X = X.loc[X.fold != fold].drop(columns=['scene', 'fold'] + [col for col in PREDICT_TARGET_COLS if not col.startswith(f"predict_{target.split('_')[0]}_")])
        train_y = y[target].loc[X.fold != fold]
        valid_X = X.loc[X.fold == fold].drop(columns=['scene', 'fold'] + [col for col in PREDICT_TARGET_COLS if not col.startswith(f"predict_{target.split('_')[0]}_")])
        valid_y = y[target].loc[X.fold == fold]

        train_data = lgb.Dataset(train_X, train_y)
        valid_data = lgb.Dataset(valid_X, valid_y, reference=train_data)

        model = lgb.train(
            lgb_params,
            train_data,
            valid_sets=[valid_data],
            num_boost_round=10000,
            callbacks=[lgb.early_stopping(100)],
        )
        oof[X.fold == fold] = model.predict(valid_X)
        models.append(model)
    return models, oof

def predict(X: pd.DataFrame, target: str, models: dict[str, list]):
    preds = []
    for model in models[target]:
        pred = model.predict(X.drop(columns=['scene'] + [col for col in PREDICT_TARGET_COLS if not col.startswith(f"predict_{target.split('_')[0]}_")]))
        preds.append(pred)
    return np.mean(preds, axis=0)

def evaluate(y_true: pd.DataFrame, y_pred: pd.DataFrame):
    return np.mean(np.abs(y_true - y_pred))

In [ ]:
train_df = preprocess(train_df)

origin_test_ids = test_df['ID'].to_pandas()
test_df = preprocess(test_df)

print(train_df.shape)
print(test_df.shape)

In [ ]:
train_df = train_df.to_pandas()
test_df = test_df.to_pandas()

In [ ]:
# Scene?????????DataFrame????
# train_df["scene"] = [i.split("_")[0] for i in train_df.ID]

# =======================================
# 5 Fold???
# =======================================

# train/val split
# Scene?Fold????
scene_list = train_df.scene.unique()
fold_map = {}

for i, scene in enumerate(scene_list):
    fold_map[scene] = i % 5

train_df["fold"] = train_df.scene.map(fold_map)
train_df.head()

In [ ]:
PREDICT_TARGET_COLS = []
for target in ['x', 'y', 'z']:
    for i in range(6):
        train_df[f"predict_{target}_{i}"] = 0
        test_df[f"predict_{target}_{i}"] = 0
        PREDICT_TARGET_COLS.append(f"predict_{target}_{i}")

oof_dir = Path('/Users/gouyashuto/localrepository/atmacup18/output/baseline52/oof')

for i in range(5):
    train_df.loc[train_df.fold == i, PREDICT_TARGET_COLS] = np.load(oof_dir / f'fold{i}_val_pred.npy')

test_df[PREDICT_TARGET_COLS] = np.load(oof_dir.parent / 'test_preds.npy').mean(axis=0)

display(train_df.head())
display(test_df.head())

In [ ]:
remove_columns = ['ID']
X = train_df.drop(columns=remove_columns + TARGET_COLS)
y = train_df[TARGET_COLS]

test_X = test_df.drop(columns=remove_columns)

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[CAT_COLS] = oe.fit_transform(X[CAT_COLS])
test_X[CAT_COLS] = oe.transform(test_X[CAT_COLS])

print(X.shape)
print(test_X.shape)

In [ ]:
%%time

models_dict = {}
oof = pd.DataFrame(np.zeros_like(train_df.loc[:, TARGET_COLS]), columns=TARGET_COLS)

preds = test_df[['ID']]
preds[TARGET_COLS] = 0.0

for target in TARGET_COLS:
    print('=' * 10, f'target: {target} start' + '=' * 10)
    partial_model, partial_oofs = train(X, y, target)
    models_dict[target] = partial_model
    oof.loc[:, target] = partial_oofs
    preds.loc[:, target] = predict(test_X, target, models_dict)

In [ ]:
for target in TARGET_COLS:
    print(f'{target} CV score: ', evaluate(y[target], oof[target]))
score = evaluate(y, oof)
print('Total CV score: ', score)

In [ ]:
submission = pd.DataFrame(origin_test_ids).merge(preds, on='ID', how='left').drop(columns=['ID'])

output_path = OUTPUT_DIR / f'{exp_name}_{score:.4f}_submission.csv'
if output_path.exists():
    assert False, f'output file already exists. {output_path}'

submission.to_csv(output_path, index=False)
submission